In [1]:
import pandas as pd
import json

# 1. Load the raw data from the data folder
file_path = '../data/raw_credit_applications.json'
with open(file_path, 'r') as file:
    data = json.load(file)

# 2. Flatten the nested 'applicant_info' so it's easy to read in a table
df = pd.json_normalize(data)

# 3. Define our Pseudonymization/Anonymization functions
def mask_ssn(ssn):
    """Masks all but the last 4 digits of an SSN to protect PII."""
    if pd.isna(ssn) or not isinstance(ssn, str):
        return ssn
    # Keep only the last 4 digits, replace the rest with 'X'
    return "XXX-XX-" + ssn[-4:] if len(ssn) >= 4 else "XXX-XX-XXXX"

def mask_ip(ip):
    """Masks the last octet of an IPv4 address for data minimization."""
    if pd.isna(ip) or not isinstance(ip, str):
        return ip
    parts = ip.split('.')
    if len(parts) == 4:
        return f"{parts[0]}.{parts[1]}.{parts[2]}.XXX"
    return ip

# 4. Apply the masks to create new secure columns
if 'applicant_info.ssn' in df.columns:
    df['applicant_info.ssn_masked'] = df['applicant_info.ssn'].apply(mask_ssn)
    
if 'applicant_info.ip_address' in df.columns:
    df['applicant_info.ip_address_masked'] = df['applicant_info.ip_address'].apply(mask_ip)

# 5. Display the "Before and After" to prove it to your professor
columns_to_show = [
    'applicant_info.full_name', 
    'applicant_info.ssn', 'applicant_info.ssn_masked', 
    'applicant_info.ip_address', 'applicant_info.ip_address_masked'
]

# Ensure we only try to display columns that actually exist in the dataframe
existing_cols = [col for col in columns_to_show if col in df.columns]

print("Privacy Demonstration: SSN and IP Address Pseudonymization")
df[existing_cols].head()

Privacy Demonstration: SSN and IP Address Pseudonymization


,applicant_info.full_name,applicant_info.ssn,applicant_info.ssn_masked,applicant_info.ip_address,applicant_info.ip_address_masked
0,Jerry Smith,596-64-4340,XXX-XX-4340,192.168.48.155,192.168.48.XXX
1,Brandon Walker,425-69-4784,XXX-XX-4784,10.1.102.112,10.1.102.XXX
2,Scott Moore,370-78-5178,XXX-XX-5178,10.240.193.250,10.240.193.XXX
3,Thomas Lee,194-35-1833,XXX-XX-1833,192.168.175.67,192.168.175.XXX
4,Brian Rodriguez,480-41-2475,XXX-XX-2475,172.29.125.105,172.29.125.XXX
